In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile
import os
import scanpy as sc
import anndata
import numpy as np
from scipy import sparse

roots = [
    Path("/data3/junyi/scvi"),
    Path("/data3/junyi/scvi_harmony10"),
    Path("/data3/junyi/scvi_harmony"),
]


def to_sparse_int(value):
    if sparse.issparse(value):
        return value.astype(np.int32, copy=False).tocsr()
    return sparse.csr_matrix(np.rint(np.asarray(value)).astype(np.int32))


def convert_h5ad(path):
    adata = anndata.read_h5ad(path)

    # 读入后删除不需要的 layer；scvi_nrom_counts 的不同后缀都匹配。
    layers_to_delete = [
        key for key in adata.layers.keys()
        if key == "count_diff"
    ]
    for key in layers_to_delete:
        del adata.layers[key]

    reconstruct_keys = [
        key for key in adata.layers.keys()
        if isinstance(key, str) and "reconstruct" in key.lower()
    ]
    for key in reconstruct_keys:
        adata.layers[key] = to_sparse_int(adata.layers[key])

    if not layers_to_delete and not reconstruct_keys:
        print(f"SKIP: {path} (没有需要处理的 layer)")
        return

    with NamedTemporaryFile(
        dir=path.parent, prefix=f".{path.name}.", suffix=".tmp", delete=False
    ) as handle:
        temporary_path = Path(handle.name)
    try:
        adata.write_h5ad(temporary_path)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)

    checked = anndata.read_h5ad(path, backed="r")
    try:
        remaining = set(checked.layers.keys())
        result = {
            key: (type(checked.layers[key]).__name__, str(checked.layers[key].dtype))
            for key in reconstruct_keys
        }
        assert not any(
            key == "count_diff"
            or (isinstance(key, str) and key.startswith("scvi_nrom_counts"))
            for key in remaining
        )
    finally:
        checked.file.close()
    print(f"DONE: {path}; deleted={layers_to_delete}; reconstruct={result}")


h5ad_files = sorted(path for root in roots for path in root.glob("*.h5ad"))
print(f"发现 {len(h5ad_files)} 个 h5ad 文件")
for h5ad_path in h5ad_files:
    convert_h5ad(h5ad_path)


发现 32 个 h5ad 文件
DONE: /data3/junyi/scvi/AMY.h5ad -> {'scvi_reconstructed_counts': ('csr_matrix', 'int32')}


KeyboardInterrupt: 

In [2]:
import scanpy as sc
adata = sc.read_h5ad("/data3/junyi/scvi_harmony/AMY_scviHarmony.h5ad")

In [ ]:
adata.obs['celltype.L0'] = adata.obs['cell.L0'].astype('category')

In [10]:
import numpy as np

In [15]:
adata.obs["celltype.L0"] = np.where(
    adata.obs["Neurotransmitter"].eq("NN"),
    "NN",
    "N",
)
adata.obs["celltype.L0"] = adata.obs["celltype.L0"].astype("category")

In [14]:
adata.obs["celltype.L0"].value_counts()

celltype.L0
N     207682
NN     39622
Name: count, dtype: int64

In [35]:
def to_sparse_int(arr):
    """把 numpy / float / 稀疏数组转换成 (n_cells, n_genes) 的 csr int 矩阵。

    - 输入若为稀疏矩阵：保持格式，只把 dtype 转成 int32。
    - 输入若为稠密 ndarray：rint 截断后构造 csr（counts 必须是离散整数）。
    - 用于把 decoder 抽样得到的 counts 写入 adata.layers["..."]，避免 h5ad
      把 float 数组按 ~8 字节/元素存储（counts 通常很稀疏 + 数值小）。
    """
    import numpy as np
    from scipy.sparse import csr_matrix, issparse
    if issparse(arr):
        return csr_matrix(arr).astype(np.int32)
    arr = np.asarray(arr)
    if arr.ndim == 1:
        # 防御性：误传 1D 时强制 reshape 成单行矩阵
        arr = arr.reshape(1, -1)
    return csr_matrix(np.rint(arr).astype(np.int32))

In [36]:
mat = to_sparse_int(adata.layers["scvi_reconstructed_counts_harmony"])

In [54]:
adata.layers['scvi_reconstructed_counts_harmony']

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 604457384 stored elements and shape (247304, 16428)>

In [55]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 606840263 stored elements and shape (247304, 16428)>

In [38]:
adata.layers["scvi_reconstructed_counts_harmony"] = mat


In [39]:
adata.layers["scvi_reconstructed_counts_harmony"]

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 604457384 stored elements and shape (247304, 16428)>

In [40]:
if "count_diff" in adata.layers:
    del adata.layers["count_diff"]

In [41]:
adata.write_h5ad("/data3/junyi/scvi_harmony/AMY.h5ad")

In [56]:
type(adata.X)
adata.X.dtype
adata.X.shape

(247304, 16428)

In [9]:
import sys

print("type:", type(mat))
print("shape:", mat.shape)
print("dtype:", mat.dtype)

if hasattr(mat, "nnz"):
    print("nnz:", mat.nnz)
    print("占用内存:", (
        mat.data.nbytes
        + mat.indices.nbytes
        + mat.indptr.nbytes
    ) / 1024**2, "MB")
else:
    print("占用内存:", mat.nbytes / 1024**2, "MB")

print("对象大小:", sys.getsizeof(mat) / 1024**2, "MB")

type: <class 'scipy.sparse._csr.csr_matrix'>
shape: (247304, 16428)
dtype: int32
nnz: 604457384
占用内存: 4612.587253570557 MB
对象大小: 4.57763671875e-05 MB


In [1]:
adata

NameError: name 'adata' is not defined